### Traveling Salesperson Problem (TSP) using VQE with Qiskit

This tutorial will guide you through solving the Traveling Salesperson Problem (TSP), a classic optimization challenge, using the Variational Quantum Eigensolver (VQE). We'll follow a standard process: setting up the TSP instance in Qiskit, converting it into a quantum-compatible format, and then applying VQE to find an approximate solution for the optimal tour.

In [ ]:
import qiskit
print(qiskit.__version__)

### Travelling Salesman Problem

The TSP on the nodes of a graph asks for the shortest **Hamiltonian cycle** that can be taken through each of the nodes. A Hamiltonian cycle is a closed path that uses every vertex of a graph once. The general solution is unknown, and an algorithm that finds it efficiently (e.g., in polynomial time) is not expected to exist.

<!-- ### Mathematical Formulation

Find the shortest Hamiltonian cycle in a graph $ G = (V, E) $ with $ n = |V| $ nodes and distances $ w_{ij} $ (distance from vertex $ i $ to vertex $ j $). A Hamiltonian cycle is described by $ N^2 $ variables $ x_{i,p} $, where $ i $ represents the node and $ p $ represents its order in a prospective cycle. The decision variable takes the value $ 1 $ if the solution occurs at node $ i $ at time order $ p $. We require that every node can only appear once in the cycle, and for each time a node has to occur. This amounts to the two constraints (here and in the following, whenever not specified, the summations run over $ 0, 1, \dots, N-1 $):

$$
\sum_i x_{i,p} = 1 \quad \forall p
$$

$$
\sum_p x_{i,p} = 1 \quad \forall i.
$$

For nodes in our prospective ordering, if $ x_{i,p} $ and $ x_{j,p+1} $ are both $ 1 $, then there should be an energy penalty if $ (i, j) \notin E $ (not connected in the graph). The form of this penalty is:

$$
\sum_{i,j \notin E} \sum_p x_{i,p} x_{j,p+1} > 0,
$$

where it is assumed the boundary condition of the Hamiltonian cycles ($ p = N $) $ \equiv (p = 0) $. However, here it will be assumed a fully connected graph and not include this term. The distance that needs to be minimized is:

$$
C(\mathbf{x}) = \sum_{i,j} w_{ij} \sum_p x_{i,p} x_{j,p+1}.
$$

Putting this all together in a single objective function to be minimized, we get the following:

$$
C(\mathbf{x}) = \sum_{i,j} w_{ij} \sum_p x_{i,p} x_{j,p+1} + A \sum_p \left( 1 - \sum_i x_{i,p} \right)^2 + A \sum_i \left( 1 - \sum_p x_{i,p} \right)^2,
$$

where $ A $ is a free parameter. One needs to ensure that $ A $ is large enough so that these constraints are respected. One way to do this is to choose $ A $ such that $ A > \max(w_{ij}) $. -->

### Setting Up the Environment

Before we define the problem, we need to import the necessary libraries. These include:

- **Basic libraries:** numpy for numerical operations and matplotlib for plotting.

- **Qiskit Optimization:** Tools to define the Knapsack problem (Knapsack), convert it into a quadratic program, and map it to a quantum problem (QuadraticProgramToQubo).

- **Qiskit Algorithms:** The core QAOA components, including the Estimator primitive for calculating expectation values and the EfficientSU2 ansatz.

- **SciPy:** A classical optimizer that QAOA will use to minimize the cost function.

- **Qiskit Aer:** To simulate a quantum backend for our experiment.

In [ ]:
# basic imports

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from docplex.mp.model import Model
# quantum imports
from qiskit_optimization.applications import Maxcut, Knapsack, Tsp
from qiskit.circuit import Parameter,QuantumCircuit
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.algorithms import CplexOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
# Pre-defined ansatz circuit and operator class for Hamiltonian
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# SciPy minimizer routine
from scipy.optimize import minimize
from qiskit.primitives import BackendEstimatorV2, BackendSamplerV2
from qiskit_aer import AerSimulator
backend = AerSimulator(method="automatic")


estimator = BackendEstimatorV2(backend=backend)
sampler = BackendSamplerV2(backend=backend)

### Generate a Random TSP Instance



In [ ]:
# set the random seed for reproducibility
seed = 135

# Generating a graph of 4 cities
n = 4
num_qubits = n**2
tsp = Tsp.create_random_instance(n, seed=seed)
adj_matrix = nx.to_numpy_array(tsp.graph)

print("=== TSP Instance Information ===")
print(f"Number of cities: {n}")
print(f"Number of qubits required (n^2): {num_qubits}")
print("\nCity coordinates:")
for idx, coord in enumerate([tsp.graph.nodes[node]["pos"] for node in tsp.graph.nodes]):
    print(f"  City {idx}: {coord}")

print("\nDistance (adjacency) matrix:")
print(adj_matrix)

print("\nCity-to-city distances:")
for i in range(n):
    for j in range(n):
        if i != j:
            print(f"  Distance from City {i} to City {j}: {adj_matrix[i, j]:.2f}")

# colors = ["r" for node in tsp.graph.nodes]
# pos = [tsp.graph.nodes[node]["pos"] for node in tsp.graph.nodes]


#### **Finding the Exact Solution Classically**

Before exploring quantum approaches like QAOA or VQE, it's essential to compute the exact optimal solution using a classical method. This provides a reliable benchmark to evaluate the performance of quantum algorithms.

We solve the TSP instance using **IBM ILOG CPLEX**, a powerful commercial optimization solver, via the `docplex` modeling API. The problem is formulated with binary decision variables $ x_{ij} $, where:
- $ x_{ij} = 1 $ if the tour goes directly from city $ i $ to city $ j $,
- $ x_{ij} = 0 $ otherwise.

To prevent disconnected sub-tours (e.g., two separate cycles), we use the **Miller-Tucker-Zemlin (MTZ) formulation**, which introduces auxiliary continuous variables $ u_i $ to enforce a single Hamiltonian cycle.


In [ ]:
# Build the model
m = Model(name='TSP_3cities')

# Decision variables
# x[i,j] = 1 if we travel directly from city i to city j
x = {}
for i in range(n):
    for j in range(n):
        if i != j:
            x[i,j] = m.binary_var(name=f"x_{i}_{j}")

# MTZ “u” variables to eliminate subtours
u = {i: m.continuous_var(lb=0, ub=n-1, name=f"u_{i}") for i in range(n)}

# Objective: minimize total travel distance
m.minimize(m.sum(adj_matrix[i,j] * x[i,j] for (i,j) in x))

# Degree constraints: leave each city once, enter each city once
for i in range(n):
    m.add_constraint(m.sum(x[i,j] for j in range(n) if j!=i) == 1, ctname=f"out_{i}")
    m.add_constraint(m.sum(x[j,i] for j in range(n) if j!=i) == 1, ctname=f"in_{i}")

# MTZ subtour‑elimination constraints (for i≠0, j≠0)
for i in range(1, n):
    for j in range(1, n):
        if i != j:
            m.add_constraint(u[i] - u[j] + n * x[i,j] <= n - 1,
                             ctname=f"mtz_{i}_{j}")

# Solve
sol = m.solve(log_output=True)
if not sol:
    print("No solution found.")
else:
    # 8. Extract the tour
    tour = [0]
    current = 0
    for _ in range(n-1):
        # find the unique j such that x[current,j] = 1
        for j in range(n):
            if current != j and sol.get_value(x[current,j]) > 0.5:
                tour.append(j)
                current = j
                break
    tour.append(0)  # return to start
    
    print("Optimal tour:", tour)
    print("Optimal cost:", m.objective_value)


In [ ]:
# ========================================
# Print Number of Classical Variables
# ========================================

# Count binary variables (x[i,j])
num_binary_vars = len(x)

# Count continuous variables (u[i])
num_continuous_vars = len(u)

# Total variables
total_variables = num_binary_vars + num_continuous_vars

print("\n📊 Classical Model Variable Count:")
print(f"   Binary variables (x[i,j]): {num_binary_vars}")
print(f"   Continuous variables (u[i]): {num_continuous_vars}")
print(f"   Total variables: {total_variables}")

#### Visualize TSP Solution

In [ ]:
def visualize_tsp_solution(G, pos, tour, title="TSP Solution Visualization", figsize=(10, 8)):
    """
    Visually highlights the optimal TSP tour on a graph with enhanced styling.
    Handles directed/undirected graphs safely.
    """
    plt.figure(figsize=figsize)
    
    # --- NODES ---
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=700, alpha=0.9, edgecolors='black')
    nx.draw_networkx_nodes(G, pos, nodelist=[tour[0]], node_shape='s', node_color='orange', node_size=800, edgecolors='black')
    nx.draw_networkx_labels(G, pos, font_size=14, font_weight='bold')

    # --- EDGES ---
    nx.draw_networkx_edges(G, pos, edge_color='gray', style='dashed', alpha=0.5, width=1)
    
    # Tour edges (circular: back to start)
    tour_edges = [(tour[i], tour[(i + 1) % len(tour)]) for i in range(len(tour))]
    nx.draw_networkx_edges(G, pos, edgelist=tour_edges, edge_color='red', width=3, alpha=0.9)

    # --- EDGE LABELS (Using adj_matrix for accuracy) ---
    edge_labels = {}
    for e in tour_edges:
        u, v = e
        weight = adj_matrix[u, v]
        edge_labels[e] = f"{weight:.1f}"

    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10, font_color='darkred')

    # --- TITLE & COST ---
    total_cost = sum(adj_matrix[e[0], e[1]] for e in tour_edges)
    plt.title(f"{title}\nTotal Tour Cost: {total_cost:.2f}", fontsize=16, fontweight='bold')

    plt.axis('equal')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Print tour summary
    print("\n📍 Tour Summary:")
    print(f"   Start City: {tour[0]}")
    print(f"   Path: {' → '.join(map(str, tour))}")
    print(f"   Total Distance: {total_cost:.2f}\n")

In [ ]:
G = tsp.graph
pos = {i: G.nodes[i]["pos"] for i in G.nodes()}
visualize_tsp_solution(G, pos, tour, title="Optimal TSP Tour (4 Cities)")

### **From Quadratic Program to QUBO**

Quantum computers solve problems formulated as Hamiltonians. A common step is to first convert the problem into a **Quadratic Unconstrained Binary Optimization (QUBO)** problem. This format represents the problem as a single quadratic equation to be minimized, without any constraints.


In [ ]:
problem = tsp.to_quadratic_program()
print(problem.prettyprint())

In [ ]:
# problem to qubo
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)
print(qubo.export_as_lp_string())

In [ ]:
num_vars = qubo.get_num_vars()
print(f"Number of variables in QUBO: {num_vars}")


### **Mapping the QUBO to an Ising Hamiltonian**

The VQE algorithm works by finding the minimum energy (eigenvalue) of a Hamiltonian. We now convert our QUBO problem into an **Ising Hamiltonian**. This Hamiltonian is an operator that can be measured on a quantum computer. Each binary variable in the QUBO is mapped to a qubit.

The conversion gives us two components:

- `qubitOp`: The Ising Hamiltonian, represented as a sum of Pauli operators (Z, ZZ).

- `offset`: A constant energy shift that we'll add back to our final result.

In [ ]:
qubitOp, offset = qubo.to_ising()
print("Offset:", offset)
print("Ising Hamiltonian:")
print(str(qubitOp))

### **The Variational Quantum Eigensolver (VQE)**

VQE is a hybrid quantum-classical algorithm. It uses a quantum computer to prepare a trial wavefunction (using a parameterized circuit called an **ansatz**) and measure the energy of the Hamiltonian. A classical computer then takes this energy and adjusts the parameters of the ansatz to find a new trial state with lower energy. This process is repeated until the minimum energy is found.

#### **The Ansatz**

The ansatz is a parameterized quantum circuit that creates the trial ground state. Its structure is crucial for the success of VQE. A good ansatz should be able to prepare the true ground state of the Hamiltonian. Here, we use `EfficientSU2`, a versatile and commonly used hardware-efficient ansatz from the Qiskit circuit library.


Let's visualize the structure of our ansatz circuit.

In [ ]:
# One shallow hardware-efficient layer is sufficient for this small instance.
reps = 1 # number of repetitions for the ansatz
ansatz = EfficientSU2(qubitOp.num_qubits, reps=reps, entanglement='linear', insert_barriers=True)  # change the entanglement strategy as needed ('full', 'linear', 'circular', etc.)
ansatz.decompose().draw("mpl", style="iqp",fold=-1)

The classical optimizer needs to tune the parameters of this ansatz. The number of parameters determines the complexity of the classical optimization task.

In [ ]:
num_params = ansatz.num_parameters
print(f"The ansatz has {num_params} trainable parameters for {ansatz.num_qubits} qubits.")

#### **Executing the VQE Algorithm**

Now we set up and run the VQE algorithm.

- **Cost Function:** We define a `cost_func` that takes the ansatz parameters, builds the circuit, and uses the `Estimator` primitive to calculate the expectation value (energy) of our Hamiltonian. This function returns the energy, which the classical optimizer will try to minimize. We also include a dictionary, `cost_history_dict`, to store the energy at each iteration so we can plot the convergence later.

- **Initial Parameters:** We start the optimization from a random initial set of parameters, `x0`.

- **Classical Optimizer:** We use the `COBYLA` optimizer from `scipy.optimize.minimize` to find the optimal parameters for our ansatz. This optimizer repeatedly calls our `cost_func` to find the parameter values that result in the lowest energy.

In [ ]:
def cost_func(params, ansatz, hamiltonian, estimator):
    """Return estimate of energy from estimator

    Parameters:
        params (ndarray): Array of ansatz parameters
        ansatz (QuantumCircuit): Parameterized ansatz circuit
        hamiltonian (SparsePauliOp): Operator representation of Hamiltonian
        estimator (EstimatorV2): Estimator primitive instance
        cost_history_dict: Dictionary for storing intermediate results

    Returns:
        float: Energy estimate
    """
    pub = (ansatz, [hamiltonian], [params])
    result = estimator.run(pubs=[pub]).result()
    energy = result[0].data.evs[0]

    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = params
    cost_history_dict["cost_history"].append(energy)
    print(f"Iters. done: {cost_history_dict['iters']} [Current cost: {energy}]")

    return energy

This dictionary will store the history of our optimization process.

In [ ]:
cost_history_dict = {
    "prev_vector": None,
    "iters": 0,
    "cost_history": [],
}

Let's generate the random starting point for the optimizer

In [ ]:
np.random.seed(0)
x0 = 2 * np.pi * np.random.random(num_params)
x0

Now, we run the classical optimizer. It will print the cost at each iteration, giving us a live view of the VQE's progress as it searches for the minimum energy.

In [ ]:
ansatz = ansatz.decompose()

res = minimize(
        cost_func,
        x0,
        args=(ansatz, qubitOp, estimator),
        method="cobyla", # You can change the method to 'nelder-mead', 'bfgs', etc. as needed
        options={"maxiter": 1000, "disp": True}, # also can adjust the options like 'maxiter', 'disp', 'rhobeg', 'tol' as needed
        tol=1e-6,  # tolerance for convergence    
    )


The optimizer has successfully terminated. The `res` object contains the results, including the final minimized energy (`fun`) and the optimal parameters (`x`).

In [ ]:
res

Now, let's plot the cost (energy) against the number of iterations. This convergence plot shows how the VQE algorithm progressively found lower energy states until it converged on a solution.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))  # Bigger figure

ax.plot(
    range(cost_history_dict["iters"]),
    cost_history_dict["cost_history"],
    color="tab:blue",
    linewidth=2,
    marker="o",
    markersize=4,
    label="VQE Energy (Objective Value)"
)

# Add annotations and explanations
ax.set_xlabel("Iteration Number", fontsize=14)
ax.set_ylabel("Energy (Cost Function Value)", fontsize=14)
ax.set_title("VQE Convergence for the TSP\n(Energy vs. Optimization Iterations)", fontsize=16, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.7)
ax.legend(fontsize=12)

# Add a text box explaining the plot
explanation = (
    "This plot shows how the VQE algorithm minimizes the cost function (energy)\n"
    "for the Travelling Salesman problem over successive optimization steps.\n"
    "Lower energy values correspond to better solutions."
)
ax.text(
    0.5, -0.18, explanation,
    fontsize=12, color="dimgray", ha="center", va="top", transform=ax.transAxes,
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray")
)

plt.tight_layout()
plt.show()

### Run the Optimized Circuit and Sample the Solution

The VQE optimization has given us the best parameters for our ansatz. We now plug these parameters into the circuit to prepare the approximate ground state of our Hamiltonian.

Since the goal is to find the solution bitstring, we add measurements to all qubits and use the `Sampler` primitive to run the circuit many times (e.g., 10,000 "shots") and collect the measurement outcomes.

In [ ]:
ansatz = ansatz.assign_parameters(res.x)
ansatz.measure_all()
ansatz.draw("mpl", style="iqp",fold=-1)

In [ ]:
pub = (ansatz,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val/shots for key, val in counts_int.items()}
final_distribution_bin = {key: val/shots for key, val in counts_bin.items()}

The output of the sampler is a probability distribution over all possible measurement outcomes (bitstrings). The bitstring with the highest probability is our candidate for the optimal solution.

In [ ]:
# Select the lowest-cost feasible sample from the measured distribution.
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]

keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
feasible_candidates = []
for key, probability in final_distribution_int.items():
    candidate_bitstring = to_bitstring(key, num_vars)
    candidate_bitstring.reverse()
    candidate_solution = converter.interpret(candidate_bitstring)
    if problem.get_feasibility_info(candidate_solution)[0]:
        candidate_cost = problem.objective.evaluate(candidate_solution)
        feasible_candidates.append((candidate_cost, probability, candidate_bitstring))

if feasible_candidates:
    _, _, most_likely_bitstring = min(feasible_candidates, key=lambda row: (row[0], -row[1]))
else:
    most_likely = keys[np.argmax(np.abs(values))]
    most_likely_bitstring = to_bitstring(most_likely, num_vars)
    most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

In [ ]:
result = converter.interpret(most_likely_bitstring)
cost = problem.objective.evaluate(result)
feasible =problem.get_feasibility_info(result)[0]


print("Result TSP:", result)
print("Result value:", cost)
print("Feasible:", feasible)

In [ ]:
print("="*40)
print("🔎 Best Known Classical Solution")
print("="*40)
print(f"  • Total Value:        {m.objective_value}")

print()

print("="*40)
print("⚛️  VQE (Quantum) Solution")
print("="*40)
print(f"  • Total Value:        {cost}")
print(f"  • Solution Vector:    {result}")
print(f"  • Feasible:           {feasible}")

if feasible:
    optimality_gap = 100 * (m.objective_value - cost) / m.objective_value
    print(f"  • Optimality Gap:     {optimality_gap:.2f}%")
else:
    print("  • Note: Quantum solution is not feasible.")

print("\n" + "-"*40)
if feasible:
    if abs(cost - m.objective_value) < 1e-6:
        print("✅ Quantum solution matches the classical optimum!")
    else:
        print("ℹ️  Quantum solution is suboptimal compared to classical.")
else:
    print("❌ Quantum solution is not feasible.")
print("-"*40)